In [2]:
import sys
sys.path.append("../ingestion/python/src")
sys.path.append("../ingestion/python/LangGraph_Agent")

from utils import *
from langgraph.graph import StateGraph, START, END
from IPython.display import Image, display

from silver_enrichment import *
from graph_silver_enrichment import *
from APIendpoint import PlacesAPI
from database import Database
import pandas as pd
import os
from dotenv import load_dotenv
load_dotenv(override=True)

llm = LLM()
places_api = PlacesAPI(os.getenv('MAPS_APP_KEY'))
db = Database()

c:\Users\rolan\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langchain_core\utils\pydantic.py:41: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1


In [2]:
r = db.execute("select * from serving.job_offer where city = 'Santiago del Estero';")
r

[{'id_offer': 'QABH-V9PfBb1b0IWAAAAAA==',
  'api_source': 'jsearch',
  'job_title': 'Data Engineer Analyst',
  'offer_description': "Blend is a premier AI services provider, committed to co-creating meaningful impact for its clients through the power of data science, AI, technology, and people. With a mission to fuel bold visions, Blend tackles significant challenges by seamlessly aligning human expertise with artificial intelligence. The company is dedicated to unlocking value and fostering innovation for its clients by harnessing world-class people and data-driven strategy.\n\nWe believe that the power of people and AI can have a meaningful impact on your world, creating more fulfilling work and projects for our people and clients. For more information, visit www.blend360.com\n\nWe are seeking a Data Engineer Analyst to contribute to our next level of growth and expansion.\n\nJob Description\n\nWhat is this position about?\n\nWe are looking for a Data Engineer Analyst to contribute t

In [5]:
from rapidfuzz import process, fuzz

# Tes données et ton input
choix_possibles = ["java"]
input_utilisateur = "python"

# extractOne renvoie un tuple : (valeur, score, index)
resultat = process.extractOne(input_utilisateur, choix_possibles, scorer=fuzz.WRatio)

valeur, score, index = resultat

print(f"Meilleure correspondance : {valeur}")
print(f"Score de similarité : {score:.2f}%")

if score >= 80:
    print("Confiance élevée : on accepte le match.")
else:
    print("Confiance faible : on demande une confirmation à l'utilisateur.")

Meilleure correspondance : java
Score de similarité : 0.00%
Confiance faible : on demande une confirmation à l'utilisateur.


In [11]:
import country_converter as coco

cc = coco.CountryConverter()
print(cc.convert(names=['AR', 'FR'], to='name_short'))
# Résultat : ['Argentina', 'France']

['Argentina', 'France']


In [3]:
db = Database()

query = """
SELECT
    id_offer as job_id,
    api_source,
    job_title,
    contract_type,
    is_remote,
    offer_languages,
    seniority,
    (
        SELECT jsonb_agg(DISTINCT val)
        FROM (
            SELECT jsonb_array_elements_text(
                COALESCE(array_to_json(skills_languages)::jsonb, '[]'::jsonb) || 
                COALESCE(array_to_json(skills_frameworks)::jsonb, '[]'::jsonb) || 
                COALESCE(array_to_json(skills_aptitudes)::jsonb, '[]'::jsonb) || 
                COALESCE(array_to_json(skills_soft)::jsonb, '[]'::jsonb) ||
                COALESCE(array_to_json(alternative_job_titles)::jsonb, '[]'::jsonb)
            ) AS val
        ) sub
    ) AS all_keywords,
    score_relevancy,
    explanation,
    company_name,
    website,
    primary_type,
    city,
    country,
    lat as latitude,   -- Aliased for Plotly map compatibility
    lon as longitude,  -- Aliased for Plotly map compatibility
    offer_url,
    published_at,
    collected_at
FROM serving.job_offer;
"""
db.execute('SELECT serving.refresh_job_offer_if_stale();')
r = db.execute(query)


In [4]:
df = pd.DataFrame(r)

In [26]:
from collections import defaultdict

def generer_index_inverse(df):
    index = defaultdict(set)
    
    for idx, row in df.iterrows():
        # 1. Nettoyage : Récupère la liste, normalise chaque mot, supprime les doublons
        keywords = row.get('all_keywords', [])
        if not isinstance(keywords, list): continue
            
        mots_uniques = {kw.lower().strip() for kw in keywords}
        
        # 2. Indexation : Ajoute l'index de l'offre à chaque mot-clé
        for kw in mots_uniques:
            index[kw].add(idx)
            
    # Convertit les sets en listes pour une manipulation facilitée
    return {k: list(v) for k, v in index.items()}

# Utilisation
index_inverse = generer_index_inverse(df)
list(index_inverse.keys())

['python',
 'dataflow',
 'cloud run',
 'google cloud platform (gcp)',
 'dataform',
 'comunicación efectiva',
 'pub/sub',
 'terraform',
 'diseñar y mantener pipelines de datos robustos y escalables',
 'entendimiento del negocio',
 'resolución de problemas',
 'sql',
 'pensamiento crítico',
 'colaborar en iniciativas apoyadas por herramientas de ia',
 'aprendizaje continuo',
 'git/gitlab',
 'bigquery',
 'colaboración con equipos multidisciplinarios',
 'ci/cd',
 'cloud storage',
 'implementar buenas prácticas de calidad, monitoreo y gobierno de datos',
 'participar en procesos de migración y modernización',
 'data engineer semi senior',
 'etl',
 'explotación de datos',
 'ingeniero de datos para tiempo indefinido - cliente del area salud',
 'carga de datos',
 'ingeniero informático',
 'análisis de datos',
 'transformación de datos',
 'modelado de datos',
 'extracción de datos',
 'data engineer',
 'azure',
 'senior data engineer',
 't‑shaped senior data engineer',
 'cloud technologies',
 'mi

In [26]:
df

,api_source,job_title,contract_type,is_remote,offer_languages,seniority,all_keywords,score_relevancy,explanation,company_name,website,primary_type,city,country,latitude,longitude,offer_url,published_at,collected_at
job_id,,,,,,,,,,,,,,,,,,,
-5_5YGoAcVzSUZGgAAAAAA==,jsearch,Data Engineer (Semi Senior),unknown,False,[es],mid,"[Aprendizaje continuo, BigQuery, CI/CD, Cloud ...",7.18,The job offer is for a Data Engineer position ...,2Brains,https://www.2brains.lat/,NaN,Santiago Metropolitan,CL,-33.431443,-70.609455,https://www.getonbrd.com/jobs/data-science-ana...,2026-06-06 01:36:30.896551+00,2026-06-10 01:36:30.896551+00
_M_aaszmcldoREuEAAAAAA==,jsearch,Ingeniero de Datos para Tiempo Indefinido - Cl...,internship,False,[es],senior,"[Análisis de Datos, Carga de Datos, ETL, Explo...",6.84,Relevancy score breakdown:\n\nScore Job (8/10)...,Abenis,NaN,NaN,NaN,NaN,NaN,NaN,https://bebee.com/cl/jobs/ingeniero-de-datos-p...,2026-05-28 01:36:30.898559+00,2026-06-10 01:36:30.898559+00
_9N8_eOMlNDGe7CyAAAAAA==,jsearch,Senior Data Engineer — Databricks,fulltime,False,[en],senior,"[Azure, Data Engineer, Senior Data Engineer, T...",7.67,Le poste de Senior Data Engineer correspond à ...,Infinite Lambda,https://infinitelambda.com,NaN,NaN,NaN,NaN,NaN,https://cl.trabajo.org/oferta-5486-c5fbdf8ddd3...,2026-06-06 01:36:30.897556+00,2026-06-10 01:36:30.897556+00
U6loljOK_B2XqcgbAAAAAA==,jsearch,Data Engineer – Spark,internship,False,[es],senior,"[Analista Big Data, Automatización de procesos...",7.44,Relevancy score breakdown:\n\nScore Job (8/10)...,Direccion Regional Metropolitana,NaN,government_office,Santiago Metropolitan,CL,-33.451096,-70.664436,https://cl.trabajo.org/oferta-5486-ff8c61091e4...,2026-06-07 01:36:30.896551+00,2026-06-10 01:36:30.896551+00
Tfisb5wQKBI3EowPAAAAAA==,jsearch,Data Engineer – Cloud,fulltime,False,[es],mid,"[Cloud Engineer, Data Engineer, ETL, SQL]",7.80,score_job,SOTRASER SCL,http://www.sotraser.cl/,transportation_service,Santiago Metropolitan,CL,-33.382429,-70.754979,https://cl.trabajo.org/oferta-5486-3bd1d69a563...,2026-06-07 01:36:30.897556+00,2026-06-10 01:36:30.897556+00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
uBj-keEXIWE0UVP3AAAAAA==,jsearch,Data Scientist Analyst Junior,fulltime,False,[es],junior,"[Análisis y atención al detalle, Data Scientis...",7.95,Relevancy score breakdown:\n\nScore Job (8/10)...,IT-TALENT Headhunter SENIOR TI,http://www.it-talenthh.com/,service,Santiago Metropolitan,CL,-33.408566,-70.548711,https://cl.trabajo.org/oferta-4039-b3bd30fb7c9...,2026-06-08 01:36:30.899559+00,2026-06-10 01:36:30.899559+00
uFYWJt8ia_fxHEYVAAAAAA==,jsearch,Data Engineer Semi Senior – Cloud Data & Migra...,fulltime,False,[],senior,"[Cloud Data & Migrations Engineer, Data Engine...",7.85,Ce poste est très intéressant car il correspon...,Confidencial,NaN,market,Santiago Metropolitan,CL,-33.453594,-70.762382,https://www.jobleads.com/cl/job/data-engineer-...,2026-06-04 01:36:30.896551+00,2026-06-10 01:36:30.896551+00
uLldMxu-653zVf3hAAAAAA==,jsearch,Junior Data Engineer,unknown,False,[],junior,"[Data Engineer, Junior Data Scientist, data an...",6.82,score_job,Practia,NaN,NaN,Montevideo,UY,-34.904568,-56.136938,https://www.practia.uy/trabajar-en-practia/jun...,2026-06-09 01:36:30.897556+00,2026-06-10 01:36:30.897556+00


In [7]:
import pycountry
_LANG_CACHE ={}
def convert_language_list(liste_codes):
    if not isinstance(liste_codes, list):
        return []
    
    noms = []
    for code in liste_codes:
        c = str(code).strip().lower()
        if not c:  # ignore les valeurs vides
            continue
        if c in _LANG_CACHE:
            noms.append(_LANG_CACHE[c])
            continue
        lang = pycountry.languages.get(alpha_2=c) or pycountry.languages.get(alpha_3=c)
        nom = lang.name if lang else c
        _LANG_CACHE[c] = nom
        noms.append(nom)
    return noms
# Ou applique sur TOUTE la colonne, pour voir le résultat global
df["test_languages"] = df["offer_languages"].apply(convert_language_list)
print(df["test_languages"].head(10))
df["offer_languages"].head(5)

0    [Spanish]
1    [Spanish]
2    [English]
3    [Spanish]
4    [Spanish]
5           []
6    [English]
7    [English]
8    [Spanish]
9    [English]
Name: test_languages, dtype: object


0    [es]
1    [es]
2    [en]
3    [es]
4    [es]
Name: offer_languages, dtype: object

In [17]:
df = pd.DataFrame(r)
df.set_index('job_id', inplace=True)

df2 = generer_index_inverse(df)
df2

NameError: name 'generer_index_inverse' is not defined

In [16]:
import country_converter as coco
import numpy as np
import pycountry
import pandas as pd
cc = coco.CountryConverter()
df = pd.DataFrame(r)
_LANG_CACHE = {}

def convert_language_list(liste_codes):
    if not isinstance(liste_codes, list):
        return []
    
    noms = []
    for code in liste_codes:
        c = str(code).strip().lower()
        if c in _LANG_CACHE:
            noms.append(_LANG_CACHE[c])
            continue
            
        lang = pycountry.languages.get(alpha_2=c) or pycountry.languages.get(alpha_3=c)
        nom = lang.name if lang else c
        _LANG_CACHE[c] = nom
        noms.append(nom)
    return noms

# 2. Appliquer la conversion uniquement sur les valeurs existantes
df.loc[mask, 'country_full'] = cc.convert(
    df.loc[mask, 'country'].tolist(), 
    to='name_short'
)

df.loc[mask_language, 'offer_languages_full'] = df.loc[mask_language, 'offer_languages'].apply(convertir_liste)
# 3. Les lignes NaN resteront NaN ou tu peux remplir par défaut
df['country_full'] = df['country_full'].replace('not found', np.nan)

In [13]:
df

,job_id,api_source,job_title,contract_type,is_remote,offer_languages,seniority,all_keywords,score_relevancy,explanation,...,primary_type,city,country,latitude,longitude,offer_url,published_at,collected_at,country_full,offer_languages_full
0,-5_5YGoAcVzSUZGgAAAAAA==,jsearch,Data Engineer (Semi Senior),unknown,False,[es],mid,"[Aprendizaje continuo, BigQuery, CI/CD, Cloud ...",7.18,The job offer is for a Data Engineer position ...,...,NaN,Santiago Metropolitan,CL,-33.431443,-70.609455,https://www.getonbrd.com/jobs/data-science-ana...,2026-06-06 01:36:30.896551+00,2026-06-10 01:36:30.896551+00,Chile,[Spanish]
1,_M_aaszmcldoREuEAAAAAA==,jsearch,Ingeniero de Datos para Tiempo Indefinido - Cl...,internship,False,[es],senior,"[Análisis de Datos, Carga de Datos, ETL, Explo...",6.84,Relevancy score breakdown:\n\nScore Job (8/10)...,...,NaN,NaN,NaN,NaN,NaN,https://bebee.com/cl/jobs/ingeniero-de-datos-p...,2026-05-28 01:36:30.898559+00,2026-06-10 01:36:30.898559+00,nan,[Spanish]
2,_9N8_eOMlNDGe7CyAAAAAA==,jsearch,Senior Data Engineer — Databricks,fulltime,False,[en],senior,"[Azure, Data Engineer, Senior Data Engineer, T...",7.67,Le poste de Senior Data Engineer correspond à ...,...,NaN,NaN,NaN,NaN,NaN,https://cl.trabajo.org/oferta-5486-c5fbdf8ddd3...,2026-06-06 01:36:30.897556+00,2026-06-10 01:36:30.897556+00,nan,[English]
3,U6loljOK_B2XqcgbAAAAAA==,jsearch,Data Engineer – Spark,internship,False,[es],senior,"[Analista Big Data, Automatización de procesos...",7.44,Relevancy score breakdown:\n\nScore Job (8/10)...,...,government_office,Santiago Metropolitan,CL,-33.451096,-70.664436,https://cl.trabajo.org/oferta-5486-ff8c61091e4...,2026-06-07 01:36:30.896551+00,2026-06-10 01:36:30.896551+00,Chile,[Spanish]
4,Tfisb5wQKBI3EowPAAAAAA==,jsearch,Data Engineer – Cloud,fulltime,False,[es],mid,"[Cloud Engineer, Data Engineer, ETL, SQL]",7.80,score_job,...,transportation_service,Santiago Metropolitan,CL,-33.382429,-70.754979,https://cl.trabajo.org/oferta-5486-3bd1d69a563...,2026-06-07 01:36:30.897556+00,2026-06-10 01:36:30.897556+00,Chile,[Spanish]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
760,uBj-keEXIWE0UVP3AAAAAA==,jsearch,Data Scientist Analyst Junior,fulltime,False,[es],junior,"[Análisis y atención al detalle, Data Scientis...",7.95,Relevancy score breakdown:\n\nScore Job (8/10)...,...,service,Santiago Metropolitan,CL,-33.408566,-70.548711,https://cl.trabajo.org/oferta-4039-b3bd30fb7c9...,2026-06-08 01:36:30.899559+00,2026-06-10 01:36:30.899559+00,Chile,[Spanish]
761,uFYWJt8ia_fxHEYVAAAAAA==,jsearch,Data Engineer Semi Senior – Cloud Data & Migra...,fulltime,False,[],senior,"[Cloud Data & Migrations Engineer, Data Engine...",7.85,Ce poste est très intéressant car il correspon...,...,market,Santiago Metropolitan,CL,-33.453594,-70.762382,https://www.jobleads.com/cl/job/data-engineer-...,2026-06-04 01:36:30.896551+00,2026-06-10 01:36:30.896551+00,Chile,[]
762,uLldMxu-653zVf3hAAAAAA==,jsearch,Junior Data Engineer,unknown,False,[],junior,"[Data Engineer, Junior Data Scientist, data an...",6.82,score_job,...,NaN,Montevideo,UY,-34.904568,-56.136938,https://www.practia.uy/trabajar-en-practia/jun...,2026-06-09 01:36:30.897556+00,2026-06-10 01:36:30.897556+00,Uruguay,[]
763,zgNIQX0lUQAqZd-XAAAAAA==,jsearch,Ingeniero de Datos - Jornada Hibrida,internship,False,[es],senior,"[Data Engineer, Excel, Ingeniero de Datos, Mic...",6.84,Relevancy score breakdown:\n\nScore Job (8/10)...,...,NaN,NaN,NaN,NaN,NaN,https://bebee.com/cl/jobs/ingeniero-de-datos-j...,2026-05-28 01:36:30.898559+00,2026-06-10 01:36:30.898559+00,nan,[Spanish]


In [2]:
import pycountry

def code_vers_langue(code_iso):
    try:
        # Recherche par code ISO 639-1 (ex: 'en') ou 639-3 (ex: 'eng')
        langue = pycountry.languages.get(alpha_2=code_iso.lower())
        if not langue:
            langue = pycountry.languages.get(alpha_3=code_iso.lower())
            
        return langue.name if langue else code_iso
    except Exception:
        return code_iso

# Test de la fonction
print(code_vers_langue("en"))  # Sortie : English
print(code_vers_langue("es"))  # Sortie : Spanish
print(code_vers_langue("fr"))  # Sortie : French


English
Spanish
French


In [38]:
df

,job_id,api_source,job_title,contract_type,is_remote,offer_languages,seniority,all_keywords,score_relevancy,explanation,...,website,primary_type,city,country,latitude,longitude,offer_url,published_at,collected_at,country_full
0,-5_5YGoAcVzSUZGgAAAAAA==,jsearch,Data Engineer (Semi Senior),unknown,False,[es],mid,"[Aprendizaje continuo, BigQuery, CI/CD, Cloud ...",7.18,The job offer is for a Data Engineer position ...,...,https://www.2brains.lat/,NaN,Santiago Metropolitan,CL,-33.431443,-70.609455,https://www.getonbrd.com/jobs/data-science-ana...,2026-06-06 01:36:30.896551+00,2026-06-10 01:36:30.896551+00,Chile
1,_M_aaszmcldoREuEAAAAAA==,jsearch,Ingeniero de Datos para Tiempo Indefinido - Cl...,internship,False,[es],senior,"[Análisis de Datos, Carga de Datos, ETL, Explo...",6.84,Relevancy score breakdown:\n\nScore Job (8/10)...,...,NaN,NaN,NaN,NaN,NaN,NaN,https://bebee.com/cl/jobs/ingeniero-de-datos-p...,2026-05-28 01:36:30.898559+00,2026-06-10 01:36:30.898559+00,NaN
2,_9N8_eOMlNDGe7CyAAAAAA==,jsearch,Senior Data Engineer — Databricks,fulltime,False,[en],senior,"[Azure, Data Engineer, Senior Data Engineer, T...",7.67,Le poste de Senior Data Engineer correspond à ...,...,https://infinitelambda.com,NaN,NaN,NaN,NaN,NaN,https://cl.trabajo.org/oferta-5486-c5fbdf8ddd3...,2026-06-06 01:36:30.897556+00,2026-06-10 01:36:30.897556+00,NaN
3,U6loljOK_B2XqcgbAAAAAA==,jsearch,Data Engineer – Spark,internship,False,[es],senior,"[Analista Big Data, Automatización de procesos...",7.44,Relevancy score breakdown:\n\nScore Job (8/10)...,...,NaN,government_office,Santiago Metropolitan,CL,-33.451096,-70.664436,https://cl.trabajo.org/oferta-5486-ff8c61091e4...,2026-06-07 01:36:30.896551+00,2026-06-10 01:36:30.896551+00,Chile
4,Tfisb5wQKBI3EowPAAAAAA==,jsearch,Data Engineer – Cloud,fulltime,False,[es],mid,"[Cloud Engineer, Data Engineer, ETL, SQL]",7.80,score_job,...,http://www.sotraser.cl/,transportation_service,Santiago Metropolitan,CL,-33.382429,-70.754979,https://cl.trabajo.org/oferta-5486-3bd1d69a563...,2026-06-07 01:36:30.897556+00,2026-06-10 01:36:30.897556+00,Chile
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
760,uBj-keEXIWE0UVP3AAAAAA==,jsearch,Data Scientist Analyst Junior,fulltime,False,[es],junior,"[Análisis y atención al detalle, Data Scientis...",7.95,Relevancy score breakdown:\n\nScore Job (8/10)...,...,http://www.it-talenthh.com/,service,Santiago Metropolitan,CL,-33.408566,-70.548711,https://cl.trabajo.org/oferta-4039-b3bd30fb7c9...,2026-06-08 01:36:30.899559+00,2026-06-10 01:36:30.899559+00,Chile
761,uFYWJt8ia_fxHEYVAAAAAA==,jsearch,Data Engineer Semi Senior – Cloud Data & Migra...,fulltime,False,[],senior,"[Cloud Data & Migrations Engineer, Data Engine...",7.85,Ce poste est très intéressant car il correspon...,...,NaN,market,Santiago Metropolitan,CL,-33.453594,-70.762382,https://www.jobleads.com/cl/job/data-engineer-...,2026-06-04 01:36:30.896551+00,2026-06-10 01:36:30.896551+00,Chile
762,uLldMxu-653zVf3hAAAAAA==,jsearch,Junior Data Engineer,unknown,False,[],junior,"[Data Engineer, Junior Data Scientist, data an...",6.82,score_job,...,NaN,NaN,Montevideo,UY,-34.904568,-56.136938,https://www.practia.uy/trabajar-en-practia/jun...,2026-06-09 01:36:30.897556+00,2026-06-10 01:36:30.897556+00,Uruguay
763,zgNIQX0lUQAqZd-XAAAAAA==,jsearch,Ingeniero de Datos - Jornada Hibrida,internship,False,[es],senior,"[Data Engineer, Excel, Ingeniero de Datos, Mic...",6.84,Relevancy score breakdown:\n\nScore Job (8/10)...,...,NaN,NaN,NaN,NaN,NaN,NaN,https://bebee.com/cl/jobs/ingeniero-de-datos-j...,2026-05-28 01:36:30.898559+00,2026-06-10 01:36:30.898559+00,NaN
